# CSCI 345 · Week 11 Lab 2 (Demo): Forecasting Electricity Demand with an RNN

**Scenario.** You work for a city's power utility. The grid operator needs to know **how much electricity the city will use in the next hour** (and, later, over the next day) so they can schedule power plants. Order too little and you get blackouts; order too much and you waste money.

This is a **sequence problem**: demand depends on what happened in the last few hours, the time of day, the day of the week, and the weather. That makes it a good fit for a recurrent neural network.

**This notebook is a complete, worked demo.** Run it top to bottom and follow the reasoning. Afterwards you'll repeat the same recipe on a new dataset in the practice notebook (`week11_lab2_practice_bikes.ipynb`).

| Step | What we do | Why it matters |
|---|---|---|
| 1 | Load and explore the data | You can't model what you don't understand |
| 2 | Turn a time series into (window → target) samples | This is how a sequence becomes RNN input |
| 3 | Split by time and normalize with training stats only | Avoids "peeking at the future" (leakage) |
| 4 | Build simple baselines | An RNN is only useful if it beats them |
| 5 | Train a many-to-one RNN with early stopping | BPTT from lecture, in practice |
| 6 | Evaluate and diagnose errors | Where and when does the model fail? |
| 7 | Experiments: which inputs and how much memory? | Connects back to long-term dependencies |
| 8 | Forecast a whole day ahead: recursive vs. direct | One-to-many: feeding predictions back in |

> 🎤 **Instructor note:** Talking points like this one are cues for the live demo. Runtime is about 1 minute in total.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": 0.3, "figure.autolayout": True})
PURPLE, GOLD, GRAY, RED = "#4B2E83", "#D4A017", "#888888", "#B01E2F"
np.random.seed(0)

### The toolkit

The next cell holds two pieces of reusable code. **You don't need to read every line now.**
- `make_energy_data()`: creates the dataset if `energy_demand.csv` isn't next to the notebook. The data is **simulated** to behave like a real city's demand: daily and weekly cycles, heating in winter, air conditioning in summer, and noise.
- `RNNRegressor`: the vanilla RNN from lecture (and from Lab 1), packaged with BPTT, the Adam optimizer, gradient clipping and early stopping. Its update rule is exactly $h_t = \tanh(x_t W_{xh} + h_{t-1} W_{hh} + b_h)$, with prediction $\hat{y} = h_T W_{hy} + b_y$ from the **last** hidden state (many-to-one).

In [ ]:
# ---------------- toolkit: data source + RNN model (run me) ----------------
def make_energy_data(n_days=365, seed=7):
    """Simulated hourly electricity demand (MW) for a mid-sized city."""
    r = np.random.default_rng(seed)
    idx = pd.date_range("2025-01-01", periods=n_days * 24, freq="h")
    hour = idx.hour.values
    doy = idx.dayofyear.values
    weekend = (idx.dayofweek.values >= 5).astype(float)
    # temperature: seasonal + daily cycle + persistent weather swings (AR(1))
    weather = np.zeros(len(idx))
    for i in range(1, len(idx)):
        weather[i] = 0.995 * weather[i - 1] + r.normal(0, 0.35)
    temp = 12 - 12 * np.cos(2 * np.pi * (doy - 15) / 365) + 4 * np.sin(2 * np.pi * (hour - 9) / 24) + weather
    # demand: daily profile (smaller on weekends) + heating/cooling + noise
    profile = (np.exp(-((hour - 9) ** 2) / 8) * 0.6 + np.exp(-((hour - 19) ** 2) / 6) * 1.0
               + np.clip(np.sin(np.pi * (hour - 6) / 16), 0, None) * 0.5)
    demand = (320 + 180 * profile * (1 - 0.25 * weekend)
              + 9.0 * np.clip(16 - temp, 0, None) + 12.0 * np.clip(temp - 22, 0, None) ** 1.3)
    noise = np.zeros(len(idx))
    for i in range(1, len(idx)):
        noise[i] = 0.7 * noise[i - 1] + r.normal(0, 8)
    demand = demand + noise
    return pd.DataFrame({"timestamp": idx, "demand_mw": demand.round(1), "temperature_c": temp.round(1)})


class RNNRegressor:
    """A vanilla RNN (many-to-one) for regression, trained with BPTT + Adam. NumPy only."""

    def __init__(self, input_size, hidden_size=32, output_size=1, seed=0):
        r = np.random.default_rng(seed)
        H = hidden_size
        self.P = {"Wxh": r.normal(0, 1 / np.sqrt(input_size), (input_size, H)),
                  "Whh": r.normal(0, 1 / np.sqrt(H), (H, H)) * 0.9,
                  "bh": np.zeros(H),
                  "Why": r.normal(0, 1 / np.sqrt(H), (H, output_size)),
                  "by": np.zeros(output_size)}
        self.H = H

    def _forward(self, X):
        B, T, _ = X.shape
        hs = [np.zeros((B, self.H))]
        for t in range(T):
            hs.append(np.tanh(X[:, t] @ self.P["Wxh"] + hs[-1] @ self.P["Whh"] + self.P["bh"]))
        return hs, hs[-1] @ self.P["Why"] + self.P["by"]

    def predict(self, X, batch=2048):
        return np.concatenate([self._forward(X[i:i + batch])[1] for i in range(0, len(X), batch)])

    def _grads(self, X, Y):
        B, T, _ = X.shape
        hs, Yhat = self._forward(X)
        err = Yhat - Y
        loss = (err ** 2).mean()
        G = {k: np.zeros_like(v) for k, v in self.P.items()}
        dY = 2 * err / err.size
        G["Why"] = hs[-1].T @ dY
        G["by"] = dY.sum(0)
        dh = dY @ self.P["Why"].T
        for t in reversed(range(T)):
            draw = (1 - hs[t + 1] ** 2) * dh
            G["bh"] += draw.sum(0)
            G["Wxh"] += X[:, t].T @ draw
            G["Whh"] += hs[t].T @ draw
            dh = draw @ self.P["Whh"].T
        return loss, G

    def fit(self, X, Y, X_val, Y_val, epochs=30, batch=64, lr=3e-3, clip=1.0, patience=5, seed=0, verbose=True):
        r = np.random.default_rng(seed)
        m = {k: np.zeros_like(v) for k, v in self.P.items()}
        v = {k: np.zeros_like(v) for k, v in self.P.items()}
        t, best, best_P, wait = 0, np.inf, None, 0
        history = {"train": [], "val": []}
        for ep in range(1, epochs + 1):
            order = r.permutation(len(X))
            losses = []
            for i in range(0, len(X), batch):
                b = order[i:i + batch]
                loss, G = self._grads(X[b], Y[b])
                norm = np.sqrt(sum((g ** 2).sum() for g in G.values()))
                if norm > clip:                                   # clip by global norm
                    G = {k: g * clip / norm for k, g in G.items()}
                t += 1
                for k in self.P:                                  # Adam
                    m[k] = 0.9 * m[k] + 0.1 * G[k]
                    v[k] = 0.999 * v[k] + 0.001 * G[k] ** 2
                    self.P[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
                losses.append(loss)
            val = ((self.predict(X_val) - Y_val) ** 2).mean()
            history["train"].append(np.mean(losses)); history["val"].append(val)
            if verbose:
                print(f"epoch {ep:>2}  train MSE {np.mean(losses):.4f}  val MSE {val:.4f}" + ("  *" if val < best else ""))
            if val < best:
                best, best_P, wait = val, {k: p.copy() for k, p in self.P.items()}, 0
            else:
                wait += 1
                if wait >= patience:
                    if verbose: print(f"early stopping: no improvement for {patience} epochs")
                    break
        self.P = best_P
        return history


def make_windows(features, target, window, horizon=1, first_row=1):
    """Slide a window over time.

    For each target hour i:
        X = features[i - window + 1 : i + 1]   (the `window` rows ending at hour i)
        Y = target[i : i + horizon]            (the value(s) we want to predict)
    Returns X (N, window, D), Y (N, horizon), and idx (the target hour of each sample).
    """
    idx = np.arange(first_row + window - 1, len(features) - horizon + 1)
    X = np.stack([features[i - window + 1: i + 1] for i in idx])
    Y = np.stack([target[i: i + horizon] for i in idx])
    return X, Y, idx
print('toolkit loaded')

---
## Step 1 · Load and explore

In [ ]:
try:
    df = pd.read_csv("energy_demand.csv", parse_dates=["timestamp"])
    print("Loaded energy_demand.csv")
except FileNotFoundError:
    df = make_energy_data()
    print("CSV not found: generated the same dataset with make_energy_data()")

print(df.shape)
df.head()

In [ ]:
df.describe().round(1)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6))
ax[0].plot(df.timestamp, df.demand_mw, lw=0.4, color=PURPLE)
ax[0].set_title("Hourly demand over one year"); ax[0].set_ylabel("MW")
ax[1].plot(df.timestamp, df.temperature_c, lw=0.4, color=GOLD)
ax[1].set_title("Temperature"); ax[1].set_ylabel("°C")
plt.tight_layout(); plt.show()

> 🎤 **Talking point:** Demand is high in **winter** (heating) and in **summer** (air conditioning), and lowest in spring and fall. So temperature matters, but not in a straight line.

Zoom in on two weeks to see the patterns within a day and within a week:

In [ ]:
two_weeks = df[(df.timestamp >= "2025-03-03") & (df.timestamp < "2025-03-17")]
plt.plot(two_weeks.timestamp, two_weeks.demand_mw, color=PURPLE)
for d in pd.date_range("2025-03-08", "2025-03-16", freq="7D"):
    plt.axvspan(d, d + pd.Timedelta(days=2), color=GOLD, alpha=0.15)
plt.title("Two weeks of demand (weekends shaded)"); plt.ylabel("MW"); plt.show()

In [ ]:
df["hour"] = df.timestamp.dt.hour
df["is_weekend"] = (df.timestamp.dt.dayofweek >= 5).astype(int)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df.groupby(["hour", "is_weekend"]).demand_mw.mean().unstack().rename(columns={0: "weekday", 1: "weekend"}).plot(ax=ax[0], color=[PURPLE, GOLD])
ax[0].set_title("Average daily profile"); ax[0].set_ylabel("MW")
ax[1].scatter(df.temperature_c, df.demand_mw, s=1, alpha=0.2, color=PURPLE)
ax[1].set_title("Demand vs. temperature"); ax[1].set_xlabel("°C")
plt.tight_layout(); plt.show()

**What we learned from exploring:**
1. A strong **daily cycle**: a morning bump and a big evening peak.
2. A **weekly cycle**: weekends are lower.
3. A **U-shaped** link to temperature: cold → heating, hot → cooling.
4. Demand changes **smoothly** from hour to hour, so the recent past is very informative.

Each of these becomes an input to our model.

---
## Step 2 · From a time series to RNN samples

An RNN needs **sequences**. We'll use a sliding window: to predict the demand at hour $t$, the RNN reads the last `WINDOW` hours, one row per hour.

Each row (one time step $x_t$) contains:

| Feature | Why |
|---|---|
| demand in the **previous** hour (a "lag") | recent history |
| temperature this hour | weather drives heating/cooling (in practice, from the weather forecast) |
| $\sin$ and $\cos$ of the hour | time of day, encoded so 23:00 and 00:00 are close together |
| is_weekend | weekly pattern |

```
 hours:     t-47   t-46   ...   t-1     t      ← the window the RNN reads (48 steps)
 input x:   [lag, temp, sin, cos, wkend] per hour
 target:                                 demand(t)
```

> ⚠️ **Leakage check:** the row for hour $t$ contains demand at $t-1$, **never** demand at $t$. Otherwise the model would be handed the answer.

In [ ]:
n = len(df)
train_end, val_end = int(0.70 * n), int(0.85 * n)          # chronological split points (Step 3)

y = df.demand_mw.values.astype(float)
temp = df.temperature_c.values.astype(float)

# Normalize with TRAINING statistics only
y_mu, y_sd = y[:train_end].mean(), y[:train_end].std()
t_mu, t_sd = temp[:train_end].mean(), temp[:train_end].std()
y_z = (y - y_mu) / y_sd

lag = np.r_[np.nan, y_z[:-1]]                               # demand one hour earlier
features = np.column_stack([
    lag,
    (temp - t_mu) / t_sd,
    np.sin(2 * np.pi * df.hour / 24),
    np.cos(2 * np.pi * df.hour / 24),
    df.is_weekend,
])
FEATURE_NAMES = ["demand_lag1", "temperature", "hour_sin", "hour_cos", "is_weekend"]
print("features:", features.shape)
pd.DataFrame(features[1:6], columns=FEATURE_NAMES).round(2)

In [ ]:
WINDOW = 48
X, Y, idx = make_windows(features, y_z, WINDOW)
print("X:", X.shape, "  ->  (samples, time steps, features per step)")
print("Y:", Y.shape, "  ->  (samples, 1 value to predict)")
print("Sample 0 predicts hour", idx[0], "=", df.timestamp[idx[0]], "using hours", idx[0] - WINDOW + 1, "to", idx[0])

---
## Step 3 · Split by time, not at random

With time series we **must not shuffle before splitting**. If we did, the model would train on Tuesday and be tested on Monday: it would be "predicting" the past from the future. Real forecasting always looks forward.

```
|------------------ train (70%) ------------------|--- val (15%) ---|--- test (15%) ---|
 Jan                                           Sep               Nov               Dec
```

- **train:** fit the weights
- **validation:** pick hyperparameters and decide when to stop training (early stopping)
- **test:** touched **once**, at the end, to report results

In [ ]:
tr = idx < train_end
va = (idx >= train_end) & (idx < val_end)
te = idx >= val_end
X_tr, Y_tr, X_va, Y_va, X_te, Y_te = X[tr], Y[tr], X[va], Y[va], X[te], Y[te]
for name, m in [("train", tr), ("val", va), ("test", te)]:
    print(f"{name:>5}: {m.sum():>5} samples   {df.timestamp[idx[m][0]]:%b %d} → {df.timestamp[idx[m][-1]]:%b %d}")

---
## Step 4 · Baselines: what does "good" look like?

Before training anything, ask: **what would a simple rule score?** We'll use the **mean absolute error (MAE)** in megawatts, which is easy to explain to a grid operator.

- **Persistence:** "next hour = this hour."
- **Seasonal naive:** "next hour = the same hour yesterday."

In [ ]:
def mae(pred, true):
    return np.mean(np.abs(pred - true))

y_true_te = y[idx[te]]
results = {
    "Persistence (last hour)": mae(y[idx[te] - 1], y_true_te),
    "Seasonal naive (same hour yesterday)": mae(y[idx[te] - 24], y_true_te),
}
for k, v in results.items():
    print(f"{k:<40} MAE = {v:6.2f} MW")

> 🎤 **Talking point:** Always report a baseline. "Our model has an MAE of 7 MW" means nothing on its own. "It cuts the error by two-thirds compared with *same hour yesterday*" is a result.

---
## Step 5 · Train the RNN

Our model:
- **input size** 5 (features per hour), **hidden size** 32, **output size** 1
- reads all 48 hours, then predicts from the **final hidden state** $h_{48}$ (many-to-one)
- loss: mean squared error on the normalized demand
- **early stopping:** after each epoch, check the validation loss and keep the best weights. Stop if it hasn't improved in 5 epochs.

In [ ]:
model = RNNRegressor(input_size=X.shape[2], hidden_size=32, output_size=1, seed=0)
n_params = sum(p.size for p in model.P.values())
print(f"Parameters: {n_params:,}. The same weights are reused at all {WINDOW} time steps.\n")

history = model.fit(X_tr, Y_tr, X_va, Y_va, epochs=30, batch=64, lr=3e-3)

In [ ]:
plt.plot(history["train"], label="train", color=PURPLE)
plt.plot(history["val"], label="validation", color=GOLD)
plt.xlabel("epoch"); plt.ylabel("MSE (normalized units)"); plt.title("Learning curves"); plt.legend(); plt.show()

> 🎤 **Talking point:** Both curves drop quickly, then flatten. If validation loss started rising while training loss kept falling, that would be **overfitting**, and early stopping would protect us from it.

---
## Step 6 · Evaluate on the test set (once!)

In [ ]:
pred_te = model.predict(X_te)[:, 0] * y_sd + y_mu          # undo the normalization to get MW
results["RNN (48-hour window)"] = mae(pred_te, y_true_te)

for k, v in results.items():
    print(f"{k:<40} MAE = {v:6.2f} MW")

plt.figure(figsize=(8, 3))
plt.barh(list(results.keys()), list(results.values()), color=[GRAY, GRAY, PURPLE])
plt.xlabel("test MAE (MW), lower is better"); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

In [ ]:
week = slice(0, 24 * 7)
ts = df.timestamp.values[idx[te]][week]
plt.plot(ts, y_true_te[week], color="black", lw=1.5, label="actual")
plt.plot(ts, pred_te[week], color=PURPLE, lw=1.2, label="RNN")
plt.plot(ts, y[idx[te] - 24][week], color=GRAY, lw=1, ls="--", label="same hour yesterday")
plt.title("One week of test-set forecasts"); plt.ylabel("MW"); plt.legend(); plt.show()

### Where does the model make mistakes?

A single number hides a lot. Break the error down by hour of day:

In [ ]:
err = pd.DataFrame({"hour": df.hour.values[idx[te]], "abs_err": np.abs(pred_te - y_true_te)})
err.groupby("hour").abs_err.mean().plot(kind="bar", color=PURPLE)
plt.title("Test MAE by hour of day"); plt.ylabel("MW"); plt.show()

> 🎤 **Talking point:** Errors are largest when demand is **changing fastest**: the morning ramp-up and the evening peak. Ask: what extra input could help at those hours?

---
## Step 7 · Experiments: what does the RNN need?

### 7a. Ablation: is temperature useful?

Train the same model **without** the temperature column and compare.

In [ ]:
def train_and_score(feature_cols, window=WINDOW, hidden=32, verbose=False):
    Xs, Ys, ids = make_windows(features[:, feature_cols], y_z, window)
    a, b, c = ids < train_end, (ids >= train_end) & (ids < val_end), ids >= val_end
    m = RNNRegressor(len(feature_cols), hidden, 1, seed=0)
    m.fit(Xs[a], Ys[a], Xs[b], Ys[b], epochs=30, verbose=verbose)
    return mae(m.predict(Xs[c])[:, 0] * y_sd + y_mu, y[ids[c]])

all_cols = [0, 1, 2, 3, 4]
no_temp = [0, 2, 3, 4]
print(f"with temperature:    MAE = {train_and_score(all_cols):.2f} MW")
print(f"without temperature: MAE = {train_and_score(no_temp):.2f} MW")

### 7b. How much history does the RNN need?

In lecture we asked how long an RNN can remember. Let's vary the window length.

In [ ]:
windows = [3, 12, 24, 48, 96]
scores = [train_and_score(all_cols, window=w) for w in windows]
plt.figure(figsize=(7, 3))
plt.plot(windows, scores, marker="o", color=PURPLE)
plt.xscale("log"); plt.xticks(windows, windows)
plt.xlabel("window length (hours)"); plt.ylabel("test MAE (MW)"); plt.title("Effect of window length")
plt.show()
for w, s in zip(windows, scores):
    print(f"window {w:>3} h: MAE {s:.2f}")

> 🎤 **Talking point:** Surprise: the window length **barely matters** here (all within about half a megawatt). For a *next-hour* forecast, last hour's demand plus this hour's temperature and time of day already carry almost all the information, so a 3-hour memory is enough. Longer windows only make BPTT chains longer and training slower. Lessons:
> - Don't assume more memory helps. **Test it**, choosing on the **validation** set. *(We peek at test here only to keep the demo short. In the practice notebook you'll use validation.)*
> - Long memory matters when the important clue is far back (the "France … French" example). Ask the class: what kind of forecasting problem would need it?

---
## Step 8 · Forecasting a whole day ahead

The operator also wants a **24-hour forecast** made at once. There are two common strategies:

**A. Recursive (feed the prediction back in).** Predict hour $t$, put that prediction into the next row as the "last hour's demand", predict $t+1$, and so on. This is the one-to-many pattern from lecture.

**B. Direct (multi-output).** Train one many-to-one RNN whose output layer has **24 numbers**, one for each of the next 24 hours.

Both use the weather forecast for the coming hours. Let's compare them against "same hour yesterday".

In [ ]:
def recursive_forecast(model, i, steps=24):
    # Forecast hours i .. i+steps-1, feeding each prediction back in as the next 'lag' input.
    win = features[i - WINDOW + 1: i + 1].copy()
    preds = []
    for k in range(steps):
        p = model.predict(win[None])[0, 0]
        preds.append(p)
        if k + 1 < steps:
            nxt = features[i + k + 1].copy()
            nxt[0] = p                         # <- our own prediction replaces the true lag
            win = np.vstack([win[1:], nxt])
    return np.array(preds) * y_sd + y_mu

# Direct model: predict the next 24 hours in one shot
X24, Y24, idx24 = make_windows(features, y_z, WINDOW, horizon=24)
a, b, c = idx24 < train_end, (idx24 >= train_end) & (idx24 < val_end - 23), idx24 >= val_end
direct = RNNRegressor(X24.shape[2], hidden_size=64, output_size=24, seed=0)
_ = direct.fit(X24[a], Y24[a], X24[b], Y24[b], epochs=40, verbose=False)

starts = np.where(c)[0][::24]                  # one forecast per test day, made at the same hour
err_rec, err_dir, err_naive = [], [], []
for s in starts:
    i = idx24[s]
    truth = y[i: i + 24]
    err_rec.append(np.abs(recursive_forecast(model, i) - truth))
    err_dir.append(np.abs(direct.predict(X24[s:s + 1])[0] * y_sd + y_mu - truth))
    err_naive.append(np.abs(y[i - 24: i] - truth))

for name, e in [("Seasonal naive", err_naive), ("Recursive RNN", err_rec), ("Direct 24-output RNN", err_dir)]:
    print(f"{name:<22} day-ahead MAE = {np.mean(e):6.2f} MW")

plt.figure(figsize=(8, 3.5))
for name, e, col in [("Seasonal naive", err_naive, GRAY), ("Recursive RNN", err_rec, RED), ("Direct RNN", err_dir, PURPLE)]:
    plt.plot(range(1, 25), np.mean(e, axis=0), marker="o", ms=3, label=name, color=col)
plt.xlabel("hours ahead"); plt.ylabel("MAE (MW)"); plt.title("Day-ahead error by forecast horizon"); plt.legend(); plt.show()

> 🎤 **Talking point:** Both RNN strategies beat "same hour yesterday" at almost every horizon. Here the **recursive** model wins. Its one-step model is very accurate, demand changes smoothly, and in our simulation the "weather forecast" is perfect. Point out the trade-off:
> - **Recursive:** one simple model, but each step feeds its **own mistakes** back in. With noisier data or an imperfect weather forecast, those errors **compound** over the 24 steps. That's the same reason RNN-generated text drifts off-topic.
> - **Direct:** no feedback loop, but one model must learn 24 different horizons at once.
>
> Neither always wins. In practice you try both and **compare on validation data**.

---
## The recipe (you'll reuse this in the practice notebook)

1. **Explore:** plot the series, zoom in, look at daily/weekly profiles and drivers.
2. **Engineer features per time step:** lagged target, known-ahead inputs (weather, calendar), cyclical time encoding.
3. **Window** the series into `(samples, time steps, features)`.
4. **Split by time** into train/val/test; **normalize with training stats only**.
5. **Baselines first:** persistence and seasonal naive.
6. **Train** with early stopping on validation.
7. **Evaluate once** on test; break errors down (by hour, by day type).
8. **Experiment** (ablations, window length, hidden size), choosing settings with **validation** data.

### Discussion questions
1. Why is it fine for the window to include *this hour's* temperature but not *this hour's* demand?
2. The test set is November–December. What could go wrong if we deployed this model in July, given it was trained on January–September?
3. Our data is simulated. Name two things real demand data has that ours doesn't (and how you'd handle them).